# High-five MRD — §4.2 / figs. 4–5

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SheffieldML/PyDeepGP/blob/main/examples/damianou2013/notebooks/04_highfive_mrd.ipynb)

Port of Matlab [`demHighFiveDemonstration.m`](https://github.com/SheffieldML/deepGP/blob/master/deepGP/matlab/demHighFiveDemonstration.m)
using `demHighFiveHgplvm1.mat` (subjects A/B, 78×62). Fits a two-view DeepGP and
compares ARD / latents to the stored hgplvm reference spaces when present.


## Setup


In [ ]:
import sys, subprocess, os, urllib.request
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
ROOT = Path.cwd().resolve()
# Prefer sibling of notebooks/ (repo layout)
for c in [ROOT.parent, ROOT, ROOT / "examples" / "damianou2013"]:
    if (c / "_common.py").is_file():
        ROOT = c
        break

if IN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        "numpy", "scipy", "matplotlib", "GPy>=1.15", "scikit-learn"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        f"git+https://github.com/SheffieldML/PyDeepGP.git@examples-damianou2013"])
    ROOT = Path.cwd().resolve()
    (ROOT / "data").mkdir(exist_ok=True)
    for dest, url in [
        ("_common.py", f"https://raw.githubusercontent.com/SheffieldML/PyDeepGP/examples-damianou2013/examples/damianou2013/_common.py"),
        ("_matlab_data.py", f"https://raw.githubusercontent.com/SheffieldML/PyDeepGP/examples-damianou2013/examples/damianou2013/_matlab_data.py"),
        ("data/hgplvmSampleDataTr1.mat",
         "https://raw.githubusercontent.com/SheffieldML/deepGP/master/deepGP/matlab/hgplvmSampleDataTr1.mat"),
        ("data/demHighFiveHgplvm1.mat",
         "https://raw.githubusercontent.com/SheffieldML/deepGP/master/deepGP/matlab/demHighFiveHgplvm1.mat"),
    ]:
        p = ROOT / dest
        if not p.is_file():
            print("Fetching", dest)
            urllib.request.urlretrieve(url, str(p))

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("Working directory:", ROOT)


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import deepgp
from _common import (
    ard_weights, dominant_dims, init_noise, unfix_noise, rbf_bias,
)
from _matlab_data import load_highfive


In [ ]:
QUICK = True
q1, q2 = (4, 3) if QUICK else (6, 4)
num_inducing = 15 if QUICK else 30
iters_fixed = 40 if QUICK else 200
iters_free = 80 if QUICK else 1500


## Load high-five views


In [ ]:
data = load_highfive()
YA = data["YA"].copy()
YB = data["YB"].copy()
YA = (YA - YA.mean(0)) / (YA.std() + 1e-12)
YB = (YB - YB.mean(0)) / (YB.std() + 1e-12)
print(f"YA={YA.shape} YB={YB.shape}")


## Fit two-view MRD DeepGP


In [ ]:
m = deepgp.DeepGP(
    [[YA.shape[1], YB.shape[1]], q1, q2],
    [YA, YB],
    kernels=[[rbf_bias(q1), rbf_bias(q1)], rbf_bias(q2)],
    num_inducing=num_inducing,
    back_constraint=False,
    inits="PCA",
)
init_noise(m, fix=True)
m.optimize(max_iters=iters_fixed, messages=False)
unfix_noise(m)
m.optimize(max_iters=iters_free, messages=False)
print(f"MRD DeepGP variational bound: {float(m.log_likelihood()):.3f}")
for i, view in enumerate(m.layers[0].views):
    print(f"View {i} ARD:", np.array2string(ard_weights(view.kern), precision=3))
print("Top ARD:", np.array2string(ard_weights(m.layers[-1].kern), precision=3))


## ARD bars (fig. 4 style)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3.2))
for i, view in enumerate(m.layers[0].views):
    w = ard_weights(view.kern)
    axes[i].bar(np.arange(1, len(w) + 1), w)
    axes[i].set_title(f"view {i} ARD (fY)")
    axes[i].set_xlabel("shared dim")
w_top = ard_weights(m.layers[-1].kern)
axes[2].bar(np.arange(1, len(w_top) + 1), w_top)
axes[2].set_title("top ARD (fX)")
fig.suptitle("High-five MRD ARD (fig. 4 style)")
fig.tight_layout()
plt.show()


## Latent projections (fig. 5 style)


In [ ]:
Xshare = np.asarray(m.layers[0].X.mean)
Xtop = np.asarray(m.layers[-1].X.mean)
d_share = dominant_dims(ard_weights(m.layers[0].views[0].kern), k=2)
d_top = dominant_dims(ard_weights(m.layers[-1].kern), k=2)

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
axes[0].plot(Xshare[:, d_share[0]], Xshare[:, d_share[1]], ".-")
axes[0].set_title("DeepGP shared (dominant)")
axes[1].plot(Xtop[:, d_top[0]], Xtop[:, d_top[1]], ".-")
axes[1].set_title("DeepGP top (dominant)")
fig.suptitle("Latent projections (fig. 5 style)")
fig.tight_layout()
plt.show()


## Reference hgplvm latents from the Matlab mat (if present)


In [ ]:
if "Xparent" in data:
    fig, axes = plt.subplots(1, 3, figsize=(10, 3.2))
    axes[0].plot(data["Xparent"][:, 0], data["Xparent"][:, 1], ".-")
    axes[0].set_title("hgplvm Xparent (ref)")
    axes[1].plot(data["Xchild1"][:, 0], data["Xchild1"][:, 1], ".-")
    axes[1].set_title("hgplvm Xchild1 (ref)")
    axes[2].plot(data["Xchild2"][:, 0], data["Xchild2"][:, 1], ".-")
    axes[2].set_title("hgplvm Xchild2 (ref)")
    fig.suptitle("Reference latents from demHighFiveHgplvm1.mat")
    fig.tight_layout()
    plt.show()
else:
    print("No Xparent/Xchild* fields in mat.")
